# INF201 – Deliverable 2
## Task 1: File Ingestion and Regular Expressions

-----

### Task 1.1 - File Ingestion

In [27]:
from pathlib import Path
import re

In [24]:
def fileread(filepath: str | Path) -> tuple[str, str]:
    """
    Read a text file using UTF-8 or Latin-1 encoding.

    Args:
        filepath (str | Path): Path to the file.

    Returns:
        tuple[str, str]: File content and the encoding used.
    """

    path = Path(filepath)

    # Try reading the file with UTF-8 encoding first
    try:
        with open(path, "r", encoding="utf-8") as file:
            content = file.read()
            encoding_used = "utf-8"

    # If UTF-8 fails, try Latin-1 encoding instead
    except UnicodeDecodeError:
        with open(path, "r", encoding="latin-1") as file:
            content = file.read()
            encoding_used = "latin-1"

    # Return both the file content and the encoding used
    return content, encoding_used

In [25]:
content, encoding_used = fileread(r"C:\infgit\Deliverable-2\data\2026\science_building\POS_01\register_sales.log")

print(content, encoding_used)

[2026-10-15 07:00:49] [POS_01] item=Croissant price=4.14 EUR (REFUND)
[2026-10-15 07:02:05] [POS_01] item=Cappuccino price=4.50 NOK (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED
[2026-10-15 07:02:26] [POS_01] item=Espresso price=2.76 USD (SUCCESS)
[2026-10-15 07:03:40] [POS_01] item=Croissant price=999.00 EUR (CANCELLED)
[2026-10-15 07:06:01] [POS_01] item=Muffin price=3.06 NOK (SUCCESS)
[2026-10-15 07:06:41] [POS_01] item=Croissant price=3.76 USD (REFUND)
[2026-10-15 07:09:10] [POS_01] item=Muffin price=3.00 NOK (REFUND)
[2026-10-15 07:11:04] [POS_01] item=Espresso price=2.59 NOK (SUCCESS)
[2026-10-15 07:12:51] [POS_01] item=Croissant price=4.10 NOK (SUCCESS)
[2026-10-15 07:15:01] [POS_01] item=Espresso price=2.96 NOK (SUCCESS)
[2026-10-15 07:15:12] [POS_01] item=Croissant price=3.70 NOK (REFUND)
[2026-10-15 07:16:56] [POS_01] item=Sandwich price=7.64 NOK (REFUND)
[2026-10-15 07:18:12] [POS_01] item=Cappuccino price=4.31 USD (SUCCESS)
[2026-10-15 07:20:28] [POS_01] item=Latte price=4.33 

### Task 1.1 – Reading files

I created a function that reads text files using UTF-8 encoding, with Latin-1 as a fallback if UTF-8 fails. The function uses `try` and `except UnicodeDecodeError` to handle different encodings and returns both the file content and the encoding used. I tested the function on a generated log file and confirmed that it successfully read the content.

### Task 1.2 - Regex and transaction management

In [26]:
import re

def parse_transactions(content: str) -> list[dict]:
    """
    Extract transactions from log file content using regex.

    Args:
        content (str): Text content from a log file.

    Returns:
        list[dict]: List of dictionaries containing transaction data.
    """

    # Create an empty list to store the transactions
    transactions = []

    # Go through the log file one line at a time
    for line in content.splitlines():

        # Search for transaction data using six capture groups
        match = re.search(
            r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s\[(\w+)\]\sitem=(\w+)\sprice=(\d+\.\d{2})\s([A-Z]{3})\s\(([A-Z]+)\)",
            line
        )

        # Only process lines that match the transaction pattern
        if match:

            # Store the extracted values in a dictionary
            values = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6)
            }

            # Add the transaction dictionary to the list
            transactions.append(values)

    # Return all extracted transactions
    return transactions      


In [22]:
transactions = parse_transactions(content)
transactions[:3]

[{'timestamp': '2026-10-15 07:00:49',
  'register_id': 'POS_01',
  'item': 'Croissant',
  'price': 4.14,
  'currency': 'EUR',
  'status': 'REFUND'},
 {'timestamp': '2026-10-15 07:02:05',
  'register_id': 'POS_01',
  'item': 'Cappuccino',
  'price': 4.5,
  'currency': 'NOK',
  'status': 'SUCCESS'},
 {'timestamp': '2026-10-15 07:02:26',
  'register_id': 'POS_01',
  'item': 'Espresso',
  'price': 2.76,
  'currency': 'USD',
  'status': 'SUCCESS'}]

### Task 1.2 – Parsing transactions

I created a function that uses regular expressions to extract transaction data from the log file. Each transaction is stored as a dictionary containing the timestamp, register ID, item, price, currency and status, with the price converted to a float. Testing showed that the function successfully extracted 80 transactions from 90 lines, correctly ignoring the 10 system messages.